# 05. portfolio: 거래 명세 → QuantLib 상품 + 엔진 → NPV

`src/quanttool/portfolio.py`

| 의존하는 것 | 제공하는 것 |
|---|---|
| `config` (관례, 엔진 선택, 템플릿), `market` (Handle, 지수) | `Portfolio`(build, npvs), `trade`, `TRADES`/`BUILDERS`/`ENGINES` |

**객체 지도** (이 순서로 본다)

1. 거래 명세: `Trade`, `Bond`, `IRSwap`
2. 템플릿: `expand`, `make_trade`, `trade`
3. `Portfolio.from_records` / `from_file`
4. 조립: `BUILDERS`
5. 엔진: `ENGINES`, `engine_name`
6. `build` → `npvs`/`npv`: 직접 계산 비교, 다른 날짜 시장, par 스왑

> 데이터는 `tests/fixtures/`의 **합성 샘플**이다. 구조를 보는 용도이고 숫자로 시장을 해석하지 않는다.

In [ ]:
from datetime import date
from pathlib import Path

import pandas as pd
import QuantLib as ql

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
FIX = ROOT / "tests" / "fixtures"  # 합성 샘플 시세·예시 북
ASOF = date(2026, 10, 23)

from quanttool.config import load_config
from quanttool.market.quotes import read_quotes

cfg = load_config(ROOT / "configs")
quotes = read_quotes(FIX / "market_sample.txt")
q = quotes[ASOF]

from quanttool.market import build_market
from quanttool.portfolio import (
    BUILDERS,
    ENGINES,
    TRADES,
    Bond,
    IRSwap,
    Portfolio,
    engine_name,
    expand,
    make_trade,
    trade,
)

market = build_market(ASOF, q, cfg)

## 거래 명세 (pydantic)
공통 `Trade`(id, product, engine) 위에 상품 종류별 필드. 시장 객체는 **이름**으로만 가리킨다.

In [ ]:
pd.DataFrame(
    {
        c.__name__: {k: str(v.annotation).replace("typing.", "") for k, v in c.model_fields.items()}
        for c in [Bond, IRSwap]
    }
)

`market_names()`: 엔진 공유 키(쓰는 시장 객체 이름). `multiplier()`: NPV에 곱할 포지션 크기.

## 템플릿: 상품 특성 + 거래 값
`expand`가 템플릿 위에 레코드를 덮고, `make_trade`가 명세 객체로, `trade`는 코드용 줄임.

In [ ]:
rec = {
    "id": "S1",
    "product": "KRW_IRS_CD91",
    "notional": 1e10,
    "fixed_rate": 0.03,
    "start_date": date(2026, 10, 26),
    "maturity_date": date(2031, 10, 26),
}
print("템플릿:", cfg.products["KRW_IRS_CD91"])
print("expand:", expand(rec, cfg.products))
t = make_trade(rec, cfg.products)
print(type(t).__name__, t.market_names(), t.multiplier())
t == trade(cfg, "KRW_IRS_CD91", **{k: v for k, v in rec.items() if k != "product"})

## `Portfolio.from_records` / `from_file`
파일은 레코드 리스트 YAML. `products=cfg.products`를 주면 템플릿을 쓴다.

In [ ]:
print((FIX / "ust_book.yaml").read_text())
book = Portfolio.from_file(FIX / "ust_book.yaml", FIX / "krw_swaps.yaml", products=cfg.products)
print(book.ids())
pd.DataFrame([t.model_dump() for t in book.trades.values()]).set_index("id")

## 조립: `BUILDERS[종류](거래, 시장, cfg)` → QuantLib 상품 (엔진은 모른다)

In [ ]:
bond = BUILDERS["Bond"](book.trades["UST_10Y"], market, cfg)
swap = BUILDERS["IRSwap"](book.trades["IRS_5Y_PAY"], market, cfg)
print(
    type(bond).__name__,
    len(bond.cashflows()),
    "현금흐름 |",
    type(swap).__name__,
    len(swap.fixedLeg()),
    len(swap.floatingLeg()),
    "(고정·변동 레그)",
)
{"TRADES": list(TRADES), "BUILDERS": list(BUILDERS), "ENGINES": list(ENGINES)}

## 엔진: `engine_name`이 고르고 `ENGINES`가 만든다
거래의 `engine` → 없으면 `products.yaml engines.defaults[종류]`. 새 방법은 `ENGINES`에 한 줄.

In [ ]:
print({i: engine_name(t, cfg) for i, t in book.trades.items()})
ENGINES["SingleCurveSwap"] = lambda t, m, p: ql.DiscountingSwapEngine(m[t.index])  # 예: CD91로 할인
single = book.trades["IRS_10Y_REC"].model_copy(
    update={"id": "IRS_10Y_SINGLE", "engine": "SingleCurveSwap"}
)

## `build(market, cfg)` → `npvs()` / `npv()`
같은 (엔진, 시장 객체) 조합은 엔진 하나를 공유한다. `npvs`는 시장의 날짜·시나리오 기준, 포지션 크기 반영.

In [ ]:
book = Portfolio([*book.trades.values(), single]).build(market, cfg)
print("엔진:", list(book.engines))
print({i: type(x).__name__ for i, x in book.instruments.items()})
display(book.npvs().round(2))
print("합계", round(book.npv(), 2))

QuantLib 직접 계산과 같다 (10Y 채권).

In [ ]:
with market.active():
    sch = ql.Schedule(
        ql.Date(15, 8, 2026),
        ql.Date(15, 8, 2036),
        ql.Period(ql.Semiannual),
        ql.UnitedStates(ql.UnitedStates.GovernmentBond),
        ql.Unadjusted,
        ql.Unadjusted,
        ql.DateGeneration.Backward,
        False,
    )
    direct = ql.FixedRateBond(
        1,
        100.0,
        sch,
        [0.04125],
        ql.ActualActual(ql.ActualActual.Bond),
        ql.Unadjusted,
        100.0,
        ql.Date(15, 8, 2026),
    )
    direct.setPricingEngine(
        ql.DiscountingBondEngine(ql.YieldTermStructureHandle(market.objects["UST"].base))
    )
    print(direct.NPV() * 1e7 / 100, book.npvs()["UST_10Y"])

같은 거래를 다른 날짜의 시장에 다시 붙이면 일간 손익.

In [ ]:
prev = build_market(date(2026, 10, 22), quotes[date(2026, 10, 22)], cfg)
now = book.npvs()
before = book.build(prev, cfg).npvs()
book.build(market, cfg)
pd.DataFrame({"10/22": before, "10/23": now, "손익": now - before}).round(2)

par 스왑(고정금리 = 시세)은 NPV 0이고 공정 금리가 시세와 같다 — 커브가 시세를 재현한다는 확인.

In [ ]:
cal = ql.SouthKorea(ql.SouthKorea.Settlement)
s = cal.advance(ql.Date(23, 10, 2026), 1, ql.Days)
start = date(s.year(), s.month(), s.dayOfMonth())
par = Portfolio(
    [
        trade(
            cfg,
            "KRW_IRS_CD91",
            id="par5Y",
            notional=1e10,
            fixed_rate=q["IRS/RATE/KRW/CD91/5Y"],
            start_date=start,
            maturity_date=start.replace(year=start.year + 5),
        )
    ]
).build(market, cfg)
with market.active():
    print(par.npv(), par.instruments["par5Y"].fairRate(), q["IRS/RATE/KRW/CD91/5Y"])

## 요약

| 객체 | 입력 | 출력 | 언제 쓰나 |
|---|---|---|---|
| `trade(cfg, 템플릿, **값)` | 템플릿 이름, 거래 값 | 명세 (`Bond`/`IRSwap`) | 코드에서 거래 만들기 |
| `Portfolio.from_file / from_records` | YAML / 레코드, `products` | `Portfolio` | 북 읽기 |
| `BUILDERS[종류]` | 명세, 시장, cfg | QuantLib 상품 | `build` 안. 새 상품은 명세 + 조립 함수 |
| `ENGINES[이름]` | 명세, 시장, 엔진 인자 | QuantLib 엔진 | `build` 안. 새 가격결정 방법 |
| `Portfolio.build(market, cfg)` | 시장, 설정 | 자기 자신 (instruments, engines) | 시장에 붙이기 (날짜마다 다시) |
| `npvs()` / `npv()` | — | Series / float | 가격, 시나리오 손익의 재료 |

## 내 메모

- 